- ## sentiment (done)
- ## satisfaction (done)
- ## کامنت منفی برای ارسال (done)
- ## مغایرت کالا 

## preprocess

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import emoji
from sklearn.utils import resample
from sentiment_analysis import SentimentAnalyzer

In [ ]:
# Download latest version

# import kagglehub
# path = kagglehub.dataset_download("radeai/basalam-comments-and-products")
# print("Path to dataset files:", path)

In [ ]:
data_path = os.getenv('DATA_PATH')

In [ ]:
# run this for first time 

reviews = pd.read_csv(f'{data_path}/BaSalam.reviews.csv')
# Balance the dataset by resampling each star rating group to have the same number of samples
description = reviews[reviews['description'].notna()]
star_groups = []
for star in description['star'].unique():
    star_groups.append(description[description['star'] == star])

min_samples = min([len(group) for group in star_groups])

balanced_samples = []
for group in star_groups:
    balanced_samples.append(resample(group, replace=False, n_samples=min(min_samples, 6000), random_state=42))

balanced_description = pd.concat(balanced_samples)
print(balanced_description['star'].value_counts())

In [ ]:
# load if you run  befor
balanced_description = pd.read_csv(f'{data_path}/labeled_comments.csv')

In [ ]:
def preprocessing(comment):
    """
    Preprocesses a given comment by performing the following steps:
    1. Replaces all emojis with a space.
    2. Removes URLs.
    3. Removes all non-word characters (punctuation).
    4. Removes all digits.

    Parameters:
    comment (str): The input comment to be preprocessed.

    Returns:
    str: The preprocessed comment.
    """
    comment = emoji.replace_emoji(comment, replace="")
    comment = re.sub(r'https?://\S+|www\.\S+', ' ', comment)
    comment = re.sub(r'[^\w\s]', ' ', comment)
    comment = comment.strip()
    # comment = re.sub(r'\d+', ' ', comment)
    return comment

In [ ]:
balanced_description['cleaned_comment'] = balanced_description['description'].map(preprocessing)

In [ ]:
balanced_description.to_csv('sample.csv', index=False)

## sentiments

In [ ]:
# sentiments
"""
This part processes a CSV file in chunks, performs sentiment analysis on each chunk,
and saves the results to a new CSV file. It uses the SentimentAnalyzer class from the
sentiment_analysis module to classify the sentiment of each description in the dataset.
"""

analyzer = SentimentAnalyzer()
chunk_size = 10
batch_size = 5

input_file = "sample.csv"
output_file = "sentiment_results.csv"

chunk_number = 0

with open(output_file, "a", encoding="utf-8") as f_out:
    for chunk in pd.read_csv(input_file, chunksize=chunk_size):
        chunk_number += 1
        sentiments = []

        print(f"Processing Chunk {chunk_number}...")

        for i in range(0, len(chunk), batch_size):
            batch = chunk.iloc[i:i + batch_size]
            descriptions = batch['cleaned_comment'].tolist()

            if descriptions:
                batch_results = analyzer.classify(descriptions, method='batch')

                if len(batch_results) == len(batch):
                    sentiments.extend(batch_results)
                else:
                    print(f"⚠️ Warning: Mismatch in batch size at Chunk {chunk_number}, Batch {i // batch_size + 1}")
                    sentiments.extend(["error"] * len(batch))

            batch_number = (i // batch_size) + 1
            print(f"   Processing Batch {batch_number} in Chunk {chunk_number}")

        if len(sentiments) == len(chunk):
            chunk["sentiment"] = sentiments
        else:
            print(f"⚠️ Error: Sentiments list ({len(sentiments)}) does not match chunk size ({len(chunk)})")
            chunk["sentiment"] = ["error"] * len(chunk)

        chunk.to_csv(f_out, mode='a', index=False, header=f_out.tell() == 0)

        print(f"✅ Finished Chunk {chunk_number}, saved results to {output_file}\n")

## satisfaction

In [ ]:
satisfaction_template = """
        You are an expert sentiment analyst. For each statement below, give a number between 0 and 1 (0 for complete dissatisfaction and 1 for complete satisfaction) that indicates the buyer's satisfaction based on the examples provided.
        Just answer  one number per sentence.

        Examples:
        sentence: "عالی بود"
        satisfaction: 1

        sentence: "سلام خیلی دیر به دستم رسید"
        satisfaction: 0

        sentence: "معمولی بود"
        satisfaction: 0.5

        sentence: "هیچ تاثیری نداشت نمیدونم چرا بعضی دروغ میگن"
        satisfaction: 0.1

        sentence: "نسبت به قیمت، کیفیتش خوبه.ممنون از غرفه دار محترم"
        satisfaction: 0.9

        sentence: "یک ماهه خراب شد"
        satisfaction: 0.3

        sentence: "خوب بود ولی درحد عالی نبود"
        satisfaction: 0.5

        sentence: "کمی سایزش مشکل داشت"
        satisfaction: 0.4
        
        sentence: "ارسال خیلی سری و خیلی هم خوبه"
        satisfaction: 1

        Now, classify these new sentences:
        {sentences}

        Return only the sentiment labels in the same order, separated by new lines.
        """

In [ ]:
# Satisfaction
"""
This part processes a CSV file in chunks, performs sentiment analysis on each chunk,
and saves the results to a new CSV file. It uses the SentimentAnalyzer class from the
sentiment_analysis module to classify the sentiment of each description in the dataset.
"""
analyzer = SentimentAnalyzer()
analyzer.update_templates(batch_template=satisfaction_template)
chunk_size = 100
batch_size = 1

input_file = "sample.csv"
output_file = "satisfaction_results.csv"

chunk_number = 0

with open(output_file, "a", encoding="utf-8") as f_out:
    for chunk in pd.read_csv(input_file, chunksize=chunk_size):
        chunk_number += 1
        satisfactions = []

        print(f"Processing Chunk {chunk_number}...")

        for i in range(0, len(chunk), batch_size):
            batch = chunk.iloc[i:i + batch_size]
            descriptions = batch['cleaned_comment'].tolist()

            if descriptions:
                batch_results = analyzer.classify(descriptions, method='batch')

                if len(batch_results) == len(batch):
                    satisfactions.extend(batch_results)
                else:
                    print(f"⚠️ Warning: Mismatch in batch size at Chunk {chunk_number}, Batch {i // batch_size + 1}")
                    satisfactions.extend(["error"] * len(batch))

            batch_number = (i // batch_size) + 1
            print(f"   Processing Batch {batch_number} in Chunk {chunk_number}")

        if len(satisfactions) == len(chunk):
            chunk["satisfaction"] = satisfactions
        else:
            print(f"⚠️ Error: Sentiments list ({len(satisfactions)}) does not match chunk size ({len(chunk)})")
            chunk["satisfaction"] = ["error"] * len(chunk)

        chunk.to_csv(f_out, mode='a', index=False, header=f_out.tell() == 0)

        print(f"✅ Finished Chunk {chunk_number}, saved results to {output_file}\n")

## delivery

##### ایا کاربر در مورد ارسال شکایت داشته

In [ ]:
delivery_template = """
        Classify the following customer review based on whether it contains a complaint about the delivery process or shipping of the product. If the review mentions issues such as late delivery, damaged package, lost item, or poor shipping service, classify it as 'True' Otherwise, classify it as 'False' Provide only the classification result.

        Examples:
        sentence: "در ارسال تاخیر داشتند متاسفانه"
        classification: True

        sentence: "به موقع ارسال شد"
        classification: False

        sentence: "لباس زیبا بود شبیه عکس ولی تاخیر بسیار زیادی داشت تا بدستم برسه"
        classification: True

        sentence: "یک هفته گذشته و هنوز سفارشم نرسیده! اصلاً از نحوه ارسال راضی نیستم."
        classification: True

        sentence: "به جای کالایی که سفارش داده بودم، چیز دیگه‌ای برام ارسال شده!"
        classification: True

        sentence: "نگ محصول کمی با عکس فرق داشت، ولی در کل راضی هستم"
        categorization: False
        
        sentence: "دو ماه هست که دارم ازش استفاده میکنم ولی هنوز نتیجه ای نگرفتم"
        categorization: False


        Now, classify these new sentences:
        {sentences}

        Return only the classification labels in the same order, separated by new lines.
        """

In [ ]:
# delivery
"""
This part processes a CSV file in chunks, performs sentiment analysis on each chunk,
and saves the results to a new CSV file. It uses the SentimentAnalyzer class from the
sentiment_analysis module to classify the sentiment of each description in the dataset.
"""
analyzer = SentimentAnalyzer()
analyzer.update_templates(batch_template=delivery_template)
chunk_size = 100
batch_size = 2

input_file = "sample.csv"
output_file = "delivery_results.csv"

chunk_number = 0

with open(output_file, "a", encoding="utf-8") as f_out:
    for chunk in pd.read_csv(input_file, chunksize=chunk_size):
        chunk_number += 1
        delivery = []

        print(f"Processing Chunk {chunk_number}...")

        for i in range(0, len(chunk), batch_size):
            batch = chunk.iloc[i:i + batch_size]
            descriptions = batch['cleaned_comment'].tolist()

            if descriptions:
                batch_results = analyzer.classify(descriptions, method='batch')

                if len(batch_results) == len(batch):
                    delivery.extend(batch_results)
                else:
                    print(f"⚠️ Warning: Mismatch in batch size at Chunk {chunk_number}, Batch {i // batch_size + 1}")
                    delivery.extend(["error"] * len(batch))

            batch_number = (i // batch_size) + 1
            print(f"   Processing Batch {batch_number} in Chunk {chunk_number}")

        if len(delivery) == len(chunk):
            chunk["delivery"] = delivery
        else:
            print(f"⚠️ Error: Sentiments list ({len(delivery)}) does not match chunk size ({len(chunk)})")
            chunk["delivery"] = ["error"] * len(chunk)

        chunk.to_csv(f_out, mode='a', index=False, header=f_out.tell() == 0)

        print(f"✅ Finished Chunk {chunk_number}, saved results to {output_file}\n")

# مغایرت کالا

In [ ]:
discrepancy_template = """
        Classify the following customer review based on whether it contains a complaint about receiving an incorrect or mismatched product. If the review mentions issues such as receiving the wrong item, a different model, incorrect size, or missing parts, classify it as 'True'. Otherwise, classify it as 'False'. Provide only the classification result.

        Examples:
        sentence: "به جای رنگ مشکی، رنگ آبی برام ارسال شده!"
        classification: True

        sentence: "دقیقاً همونی بود که سفارش داده بودم، خیلی راضی‌ام."
        classification: False

        sentence: "کفشی که برام فرستادن یه سایز بزرگ‌تر از چیزی که سفارش دادم!"
        classification: True

        sentence: "مدل گوشی که دریافت کردم با چیزی که تو سایت بود فرق داره."
        classification: True

        sentence: "جنس همونیه که توی توضیحات نوشته شده بود، کیفیتش خوبه."
        classification: False

        sentence: "متأسفانه به جای پیراهن مردانه، یه پیراهن زنانه فرستادن!"
        classification: True

        sentence: "کیفیتش بد نیست ولی انتظار بیشتری داشتم."
        classification: False

        Now, classify these new sentences:
        {sentences}

        Return only the classification labels in the same order, separated by new lines.
        """

In [ ]:
# discrepancy
"""
This part processes a CSV file in chunks, performs sentiment analysis on each chunk,
and saves the results to a new CSV file. It uses the SentimentAnalyzer class from the
sentiment_analysis module to classify the sentiment of each description in the dataset.
"""
analyzer = SentimentAnalyzer(model='pixtral-large-latest')
analyzer.update_templates(batch_template=discrepancy_template)
chunk_size = 50
batch_size = 2

input_file = "sample.csv"
output_file = "discrepancy_results.csv"

chunk_number = 0

with open(output_file, "a", encoding="utf-8") as f_out:
    for chunk in pd.read_csv(input_file, chunksize=chunk_size):
        chunk_number += 1
        discrepancy = []

        print(f"Processing Chunk {chunk_number}...")

        for i in range(0, len(chunk), batch_size):
            batch = chunk.iloc[i:i + batch_size]
            descriptions = batch['cleaned_comment'].tolist()

            if descriptions:
                batch_results = analyzer.classify(descriptions, method='batch')

                if len(batch_results) == len(batch):
                    print(batch_results, len(batch))
                    discrepancy.extend(batch_results)
                else:
                    print(batch_results, len(batch))
                    print(f"⚠️ Warning: Mismatch in batch size at Chunk {chunk_number}, Batch {i // batch_size + 1}")
                    discrepancy.extend(["Error"] * len(batch))

            batch_number = (i // batch_size) + 1
            print(f"   Processing Batch {batch_number} in Chunk {chunk_number}")

        if len(discrepancy) == len(chunk):
            chunk["discrepancy"] = discrepancy
        else:
            print(f"⚠️ Error: Sentiments list ({len(discrepancy)}) does not match chunk size ({len(chunk)})")
            chunk["discrepancy"] = ["error"] * len(chunk)

        chunk.to_csv(f_out, mode='a', index=False, header=f_out.tell() == 0)

        print(f"✅ Finished Chunk {chunk_number}, saved results to {output_file}\n")